# NeuroDetect Model 3 – PROCESS-2 (CTD) Speech Classification: HC / MCI / Dementia  —  FINAL full-training notebook
## 1. Project Overview and Model 3 Architecture
Two independent branches + late fusion, trained on **PROCESS-2, Cookie Theft Description (CTD) only**, one sample per **participant**.
* **Branch A (audio):** frozen `facebook/wav2vec2-base` mean-pooled embeddings (Exp 1) + interpretable acoustic features (Exp 2).
* **Branch B (transcript):** frozen `distilbert-base-uncased` embeddings + linguistic features (Exp 3).
* **Exp 4 – late fusion:** logistic regression on branch probabilities. Branch hyper-parameters, the fusion variant and the fusion `C` are all selected **inside** each outer cross-validation fold (nested, fold-contained – see §22–23), so the fusion CV estimate does not rely on hyper-parameters tuned on its own validation fold. The official test partition is evaluated once, after model selection is frozen.

> **NeuroDetect Model 3 is an experimental speech-based research screening model. Its predictions are not a medical diagnosis and must not be used as a substitute for professional clinical assessment.**

Models 1/2 (cognitive games, PHQ-9/GAD-7) and the old Model 3A/3B are **not** used here. No metrics are pre-filled: every number comes from your run.

**Run order:** this is the **final** notebook – `MODE = "full"` is already set in the config cell (§3), so *all eligible CTD participants* are used. Just run all cells top to bottom. `MODE = "debug"` (tiny subset, separate cache, `DEBUG`-labelled package that can never be exported as the final ZIP) is kept only for future pipeline testing. Slowest cells: §13 (Wav2Vec2 + pitch extraction), §16 (DistilBERT) and §19–22 (nested CV). Extraction is cached and resumable; nested CV is not cached (re-run if the runtime restarts after §22).

## 2. Research Scope and Dataset Usage Restrictions
PROCESS-2 is access-controlled. The Hugging Face account whose token is used **must have accepted the PROCESS-2 Data Use Agreement** (this need not be the same as the Google account running Colab). This notebook: uses the data only for research/education; does **not** identify participants or speakers; prints **no transcripts, audio or participant IDs**; caches only derived numeric features (embeddings/statistics) under a restricted local folder; exports **no** recordings, transcripts, tokens or IDs. It never prints, stores or exports the Hugging Face token. Do not commit the cache/work folder to GitHub. Age/gender are never used as predictors. SVM/RandomForest files contain derived feature vectors, not raw data – still treat the package as research-restricted.

## 3. Environment Setup and Dependency Installation

In [ ]:
import sys, subprocess
PKGS = ["datasets>=3.0", "transformers>=4.40", "huggingface_hub", "librosa>=0.10", "soundfile", "scikit-learn>=1.3",
        "joblib", "pandas", "matplotlib", "tqdm", "scipy"]
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], check=True)
except Exception as e:
    raise RuntimeError(f"Dependency install failed ({e}). Check internet access / pip, or install: {PKGS}")

In [ ]:
%%writefile predict_model3.py
"""NeuroDetect Model 3 (PROCESS-2 / CTD): shared feature extraction + inference.
Used by BOTH the training notebook and inference, so preprocessing is identical.
Experimental research screening model - NOT a medical diagnosis."""
import io, json, re
from pathlib import Path
import numpy as np

W2V_ID = "facebook/wav2vec2-base"
BERT_ID = "distilbert-base-uncased"
SR, CHUNK_S, MIN_TAIL_S = 16000, 30, 1.0
TOP_DB, PAUSE_MIN_S = 30, 0.25
FILLERS = {"um", "uh", "er", "erm", "hmm", "ah", "uhm", "mm"}
PRONOUNS = {"i", "me", "my", "mine", "you", "your", "he", "him", "his", "she", "her", "hers", "it", "its",
            "we", "us", "our", "they", "them", "their", "this", "that", "these", "those"}
SUBORD = {"and", "but", "because", "which", "that", "when", "while", "if", "so", "although", "who", "where"}
DISCLAIMER = ("NeuroDetect Model 3 is an experimental speech-based research screening model. Its predictions are "
              "not a medical diagnosis and must not be used as a substitute for professional clinical assessment.")
ACOUSTIC_NAMES = (["duration_s", "speaking_ratio", "silence_total_s", "pause_count", "pause_mean_s",
                   "pause_rate_per_min", "rms_mean", "rms_std", "rms_p95", "f0_mean", "f0_std", "f0_median",
                   "f0_range_p5_p95", "voiced_fraction", "words_per_min", "articulation_rate_wpm"]
                  + [f"mfcc{i}_{s}" for i in range(13) for s in ("mean", "std")])
LING_NAMES = ["word_count", "sentence_count", "avg_sentence_len", "type_token_ratio", "mattr_50", "unique_words",
              "repeated_word_count", "filler_count", "filler_ratio", "pronoun_ratio", "mean_word_len",
              "subordination_ratio"]


def get_device():
    import torch
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


# ---------------- audio ----------------
def _read(src):
    import soundfile as sf, librosa
    try:
        y, sr = sf.read(src, dtype="float32", always_2d=True)
        return y.mean(1), sr
    except Exception:
        if hasattr(src, "seek"):
            src.seek(0)
        y, sr = librosa.load(src, sr=None, mono=True)
        return y, sr


def load_audio(obj):
    """Return (mono float32, sample_rate). Handles datasets AudioDecoder, dict(array|bytes|path), file path."""
    if obj is None:
        raise ValueError("audio is missing")
    if hasattr(obj, "get_all_samples"):  # torchcodec AudioDecoder
        s = obj.get_all_samples()
        y = s.data.cpu().numpy() if hasattr(s.data, "cpu") else np.asarray(s.data)
        sr = s.sample_rate
    elif isinstance(obj, dict):
        if obj.get("array") is not None:
            y, sr = np.asarray(obj["array"]), obj["sampling_rate"]
        elif obj.get("bytes"):
            y, sr = _read(io.BytesIO(obj["bytes"]))
        elif obj.get("path"):
            y, sr = _read(obj["path"])
        else:
            raise ValueError("audio dict has no array/bytes/path")
    elif isinstance(obj, (str, Path)):
        y, sr = _read(str(obj))
    else:
        raise TypeError(f"unsupported audio type {type(obj)}")
    y = np.asarray(y, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=int(np.argmin(y.shape)))
    return y, int(sr)


def prep_audio(y, sr):
    """Resample to 16 kHz, remove NaNs, peak-normalise. Raises on silent/too-short audio."""
    import librosa
    y = np.nan_to_num(np.asarray(y, dtype=np.float32))
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    if len(y) < 0.5 * SR:
        raise ValueError("audio shorter than 0.5 s")
    m = float(np.max(np.abs(y)))
    if m < 1e-6:
        raise ValueError("audio is silent")
    return (y / m).astype(np.float32)


def acoustic_features(y, transcript=None):
    """Interpretable acoustic features. Anything not reliably computable stays NaN (imputed inside CV folds)."""
    import librosa
    f = {k: np.nan for k in ACOUSTIC_NAMES}
    dur = len(y) / SR
    f["duration_s"] = dur
    speech = np.nan
    try:
        iv = librosa.effects.split(y, top_db=TOP_DB, frame_length=1024, hop_length=256)
        speech = float(sum(e - s for s, e in iv)) / SR
        f["speaking_ratio"], f["silence_total_s"] = speech / dur, dur - speech
        gaps = (iv[1:, 0] - iv[:-1, 1]) / SR if len(iv) > 1 else np.array([])
        p = gaps[gaps >= PAUSE_MIN_S]
        f["pause_count"], f["pause_rate_per_min"] = float(len(p)), len(p) / (dur / 60)
        f["pause_mean_s"] = float(p.mean()) if len(p) else np.nan
    except Exception:
        pass
    try:
        r = librosa.feature.rms(y=y, frame_length=1024, hop_length=256)[0]
        f["rms_mean"], f["rms_std"], f["rms_p95"] = float(r.mean()), float(r.std()), float(np.percentile(r, 95))
    except Exception:
        pass
    try:
        f0, vf, _ = librosa.pyin(y, fmin=65, fmax=400, sr=SR, frame_length=1024)
        v = f0[~np.isnan(f0)]
        f["voiced_fraction"] = float(np.mean(vf))
        if len(v) >= 10:
            f["f0_mean"], f["f0_std"], f["f0_median"] = float(v.mean()), float(v.std()), float(np.median(v))
            f["f0_range_p5_p95"] = float(np.percentile(v, 95) - np.percentile(v, 5))
    except Exception:
        pass
    try:
        m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=13)
        for i in range(13):
            f[f"mfcc{i}_mean"], f[f"mfcc{i}_std"] = float(m[i].mean()), float(m[i].std())
    except Exception:
        pass
    n = len(_words(clean_text(transcript))) if transcript else 0
    if n:
        f["words_per_min"] = n / dur * 60
        if speech and speech > 0:
            f["articulation_rate_wpm"] = n / speech * 60
    return f


# ---------------- text ----------------
def clean_text(t):
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return ""
    t = re.sub(r"\[[^\]]*\]|<[^>]*>|\{[^}]*\}", " ", str(t))
    return re.sub(r"\s+", " ", t).strip()


def _words(text):
    return re.findall(r"[a-z']+", text.lower())


def linguistic_features(text):
    f = {k: np.nan for k in LING_NAMES}
    text = clean_text(text)
    w = _words(text)
    n = len(w)
    if n == 0:
        return f
    ns = max(len([s for s in re.split(r"[.!?]+", text) if _words(s)]), 1)
    f.update(word_count=n, sentence_count=ns, avg_sentence_len=n / ns, type_token_ratio=len(set(w)) / n,
             unique_words=len(set(w)), repeated_word_count=sum(a == b for a, b in zip(w, w[1:])),
             filler_count=sum(x in FILLERS for x in w), filler_ratio=sum(x in FILLERS for x in w) / n,
             pronoun_ratio=sum(x in PRONOUNS for x in w) / n, mean_word_len=float(np.mean([len(x) for x in w])),
             subordination_ratio=sum(x in SUBORD for x in w) / n)  # proxy for syntactic complexity (no parser)
    if n >= 50:
        f["mattr_50"] = float(np.mean([len(set(w[i:i + 50])) / 50 for i in range(n - 49)]))
    return f


# ---------------- frozen encoders ----------------
class Encoders:
    def __init__(self, device=None, revisions=None, batch_size=8):
        self.device, self.rev, self.bs = device or get_device(), revisions or {}, batch_size
        self._w = self._t = None

    def audio_embedding(self, y):
        """Mean-pooled frozen Wav2Vec2 embedding (768-d); long audio is chunked (30 s) and length-weight averaged."""
        import torch
        if self._w is None:
            from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
            r = self.rev.get(W2V_ID)
            self._wfe = Wav2Vec2FeatureExtractor.from_pretrained(W2V_ID, revision=r)
            self._w = Wav2Vec2Model.from_pretrained(W2V_ID, revision=r).to(self.device).eval()
            for p in self._w.parameters():
                p.requires_grad = False
        n = int(CHUNK_S * SR)
        ch = [y[i:i + n] for i in range(0, len(y), n)]
        if len(ch) > 1 and len(ch[-1]) < MIN_TAIL_S * SR:
            ch = ch[:-1]
        full, rest = [c for c in ch if len(c) == n], [c for c in ch if len(c) != n]
        groups = [full[i:i + self.bs] for i in range(0, len(full), self.bs)] + [[c] for c in rest]
        embs, ws = [], []
        with torch.inference_mode():
            for g in groups:
                x = self._wfe(g, sampling_rate=SR, return_tensors="pt", padding=False)["input_values"].to(self.device)
                h = self._w(x).last_hidden_state.mean(1).float().cpu().numpy()
                embs += list(h)
                ws += [len(c) for c in g]
        return np.average(np.stack(embs), axis=0, weights=ws).astype(np.float32)

    def text_embedding(self, text):
        """Mean-pooled frozen DistilBERT embedding (768-d); long text uses 512-token overlapping windows. None if empty."""
        import torch
        text = clean_text(text)
        if not text:
            return None
        if self._t is None:
            from transformers import AutoTokenizer, AutoModel
            r = self.rev.get(BERT_ID)
            self._tok = AutoTokenizer.from_pretrained(BERT_ID, revision=r)
            self._t = AutoModel.from_pretrained(BERT_ID, revision=r).to(self.device).eval()
            for p in self._t.parameters():
                p.requires_grad = False
        enc = self._tok(text, truncation=True, max_length=512, stride=64, return_overflowing_tokens=True,
                        padding=True, return_tensors="pt")
        ids, mask = enc["input_ids"], enc["attention_mask"]
        out = []
        with torch.inference_mode():
            for i in range(0, len(ids), self.bs):
                a, m = ids[i:i + self.bs].to(self.device), mask[i:i + self.bs].to(self.device)
                h = self._t(input_ids=a, attention_mask=m).last_hidden_state
                mm = m.unsqueeze(-1).float()
                out.append(((h * mm).sum(1) / mm.sum(1)).float().cpu().numpy())
        return np.concatenate(out).mean(0).astype(np.float32)


# ---------------- inference ----------------
class NeuroDetectModel3:
    def __init__(self, package_dir=None, device=None, batch_size=8):
        import joblib
        self.dir = Path(package_dir) if package_dir else Path(__file__).resolve().parent.parent
        self.cfg = json.load(open(self.dir / "preprocessing" / "model_config.json"))
        lm = json.load(open(self.dir / "preprocessing" / "label_mapping.json"))
        self.labels = {int(k): v for k, v in lm["id_to_label"].items()}
        self.clf = {k: joblib.load(self.dir / "models" / f) for k, f in self.cfg["model_files"].items()}
        self.enc = Encoders(device, self.cfg.get("encoder_revisions"), batch_size)
        self.version = self.cfg["model_version"]

    def _branches(self):
        c = self.cfg
        return list(c["fusion_inputs"]) if c["final_model"] == "fusion" else [c["final_model"]]

    def required_inputs(self):
        b = set(self._branches())
        return {"audio": bool(b & {"audio_embedding", "acoustic"}), "transcript": "transcript" in b}

    def predict(self, audio_path=None, transcript=None):
        need, errs = self.required_inputs(), []
        if need["audio"] and (not audio_path or not Path(audio_path).is_file()):
            errs.append("audio_path missing or file not found (audio is required by this model)")
        if need["transcript"] and not clean_text(transcript):
            errs.append("transcript missing or empty (transcript is required by this model)")
        if errs:
            return {"status": "error", "errors": errs, "model_version": self.version, "disclaimer": DISCLAIMER}
        y = prep_audio(*load_audio(audio_path)) if need["audio"] else None
        probs = {}
        for b in self._branches():
            if b == "audio_embedding":
                x = self.enc.audio_embedding(y)[None]
            elif b == "acoustic":
                a = acoustic_features(y, transcript)
                x = np.array([[a[k] for k in self.cfg["acoustic_feature_names"]]], dtype=float)
            else:
                e = self.enc.text_embedding(transcript)
                l = linguistic_features(transcript)
                x = np.concatenate([e, [l[k] for k in self.cfg["linguistic_feature_names"]]])[None].astype(float)
            probs[b] = self.clf[b].predict_proba(x)[0]
        if self.cfg["final_model"] == "fusion":
            final = self.clf["fusion"].predict_proba(np.hstack([probs[b] for b in self.cfg["fusion_inputs"]])[None])[0]
            classes = self.clf["fusion"].classes_
        else:
            b = self.cfg["final_model"]
            final, classes = probs[b], self.clf[b].classes_
        d = {self.labels[int(c)]: float(p) for c, p in zip(classes, final)}
        return {"status": "ok", "predicted_class": max(d, key=d.get), "probabilities": d,
                "branch_probabilities": {b: {self.labels[i]: float(p) for i, p in enumerate(v)} for b, v in probs.items()},
                "components_used": self._branches(), "model_version": self.version, "disclaimer": DISCLAIMER}


if __name__ == "__main__":
    import sys
    m = NeuroDetectModel3()
    print(json.dumps(m.predict(sys.argv[1] if len(sys.argv) > 1 else None, sys.argv[2] if len(sys.argv) > 2 else None), indent=2))

In [ ]:
import os, json, random, warnings, pickle, shutil, time, zipfile, platform, hashlib, importlib, re, sys
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Apple-silicon (MPS): unsupported ops fall back to CPU instead of crashing
sys.path.insert(0, os.getcwd())
import predict_model3 as pm          # shared feature code (identical to inference)
importlib.reload(pm)

# ------------------------- CONFIG -------------------------
IN_COLAB = "google.colab" in sys.modules
MODE = "full"                   # "full" = FINAL run on ALL eligible CTD participants | "debug" = tiny subset, pipeline testing only
SEED = 42
BATCH_SIZE = 4                  # encoder batch size (30 s audio chunks / 512-token windows). Lower it if you hit out-of-memory; it does not change the cache key.
N_SPLITS = 5                    # OUTER grouped CV folds (training partition only)
DEBUG_PER_CLASS = 8             # used ONLY when MODE == "debug"
INNER_SPLITS = N_SPLITS         # INNER grouped CV folds for hyper-parameter selection inside each outer fold
REQUIRE_COMPLETE_MODALITIES = True   # exclude (and report) participants with no usable CTD audio or transcript instead of imputing a whole missing modality
MIN_TEST_PER_CLASS_WARN = 10    # print a warning if a test class has fewer participants than this
RESET_CACHE = False             # True = delete this run's feature cache first
OUTPUT_DIR = Path("/content" if IN_COLAB else "./neurodetect_output").resolve()   # ZIP is written here
COLUMN_OVERRIDES = {}           # meta-info.csv overrides if auto-detect fails, e.g. {"id":"participant_id","label":"diagnosis","split":"split"}
LABEL_OVERRIDES = {}            # raw label value -> "HC" | "MCI" | "Dementia"
DROP_LABELS = []                # raw diagnosis values to exclude (not HC/MCI/Dementia)
CTD_TASK_VALUES = None          # (unused: CTD is selected by file name)
PARTITION_COLUMN, TEST_VALUES = None, ["test"]   # None = auto-detect a split column in meta-info.csv
# -----------------------------------------------------------
assert MODE in ("full", "debug"), f"MODE must be 'full' or 'debug', got {MODE!r}"
RESULT_TAG = "[DEBUG - NOT MEANINGFUL] " if MODE == "debug" else ""
WORK_DIR = OUTPUT_DIR / "work"; CACHE_ROOT = WORK_DIR / "cache" / MODE     # separate cache tree per mode; a fingerprint sub-folder is added in section 13
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print("=" * 72)
print(f"MODE = {MODE.upper()}  |  output: {OUTPUT_DIR}")
print("FULL training: every eligible CTD participant is used; cache + package are marked FULL." if MODE == "full"
      else "DEBUG: tiny subset for pipeline testing ONLY - results are meaningless and the package cannot be exported as the final ZIP.")
print("=" * 72)

## 4. Device Detection

In [ ]:
import numpy as np, pandas as pd, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = pm.get_device()      # cuda > mps > cpu
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU/MPS found - Wav2Vec2 extraction will be much slower. In Colab: Runtime > Change runtime type > GPU. "
          "The run still works on CPU (use small BATCH_SIZE; extraction is resumable).")
def _rev(m, tries=3):
    from huggingface_hub import model_info
    err = None
    for i in range(tries):
        try: return model_info(m).sha
        except Exception as e: err = e; time.sleep(2 ** i)
    if MODE == "full":     # a reproducible FINAL run needs pinned encoder revisions
        raise RuntimeError(f"Cannot resolve the exact revision of {m} ({type(err).__name__}). Check internet / Hugging Face access and re-run.")
    print("Could not resolve revision for", m); return None
REV = {pm.W2V_ID: _rev(pm.W2V_ID), pm.BERT_ID: _rev(pm.BERT_ID)}
print(REV)

## 5. PROCESS-2 Dataset Loading
`load_dataset()` misreads this repo: it is a folder tree (`PROCESS-2_rec__NNN/…__CTD|SFT|PFT.wav/.txt`) plus `meta-info.csv`, so its auto-generated `label` is just the participant folder. The notebook therefore reads the raw files with `huggingface_hub` (lazy per-file download; only CTD files are fetched), **pinned to one dataset commit** so every file comes from the same snapshot (the commit hash is part of the cache key). Authentication: Colab secret `HF_TOKEN` (notebook access ON), the `HF_TOKEN` environment variable, or an interactive prompt – the token is never printed or exported. Execution stops with a clear error if the dataset cannot be loaded.

In [ ]:
import re
from huggingface_hub import login, list_repo_files, hf_hub_download, HfApi
tok = os.environ.get("HF_TOKEN")
if not tok and IN_COLAB:
    try:
        from google.colab import userdata; tok = userdata.get("HF_TOKEN")
    except Exception: tok = None
login(token=tok, add_to_git_credential=False) if tok else login()     # interactive prompt if no secret/env token
del tok                                                              # token is never printed, stored in a variable, or exported
REPO = "CognoSpeak/PROCESS-2"
try:
    DATASET_REV = HfApi().dataset_info(REPO).sha                     # pin ONE dataset snapshot
    assert DATASET_REV
    FILES = list_repo_files(REPO, repo_type="dataset", revision=DATASET_REV)
except Exception as e:
    raise RuntimeError("Cannot access CognoSpeak/PROCESS-2. This Hugging Face account must have accepted the PROCESS-2 Data Use Agreement "
                       "(dataset page), the token needs read access, and the runtime needs internet. Full training cannot continue.") from e
def dl(f, tries=4):
    """Download one dataset file (cached local path). Retries transient errors, then stops with a clear error - never returns a fake/empty result."""
    for i in range(tries):
        try: return hf_hub_download(REPO, f, repo_type="dataset", revision=DATASET_REV)
        except Exception as e:
            if i == tries - 1: raise RuntimeError(f"Could not download a PROCESS-2 file after {tries} attempts ({type(e).__name__}). Check access/connectivity and re-run (downloads resume).") from e
            time.sleep(2 ** i)
if "meta-info.csv" not in FILES: raise RuntimeError("meta-info.csv not found in the dataset repository - unexpected layout, stopping.")
rx = re.compile(r"^(PROCESS-2_rec__\d+)/\1__([A-Z]+)\.(wav|txt)$")
REC = {}
for f in FILES:
    m_ = rx.match(f)
    if m_: REC.setdefault(m_.group(1), {}).setdefault(m_.group(2), {})[m_.group(3)] = f
from collections import Counter
if not REC: raise RuntimeError("No participant folders matched the expected PROCESS-2 layout - stopping.")
print("authenticated; dataset commit:", DATASET_REV[:12], "| participant folders:", len(REC), "| files per task:", dict(Counter(t for r in REC.values() for t in r)))

## 6. Dataset Schema Inspection
Raw layout: one folder per participant, one wav + one txt transcript per task. Participant metadata (incl. diagnosis) comes from `meta-info.csv`. Only column names, counts and low-cardinality value counts are printed; transcript structure is shown masked.

In [ ]:
meta_csv = pd.read_csv(dl("meta-info.csv"))
print("meta-info.csv shape:", meta_csv.shape); print("columns:", list(meta_csv.columns)); print("dtypes:", meta_csv.dtypes.astype(str).to_dict())
print("missing values:", {c: int(n) for c, n in meta_csv.isna().sum().items() if n} or "none")
for c in meta_csv.columns:
    n = meta_csv[c].nunique(dropna=True)
    print(f"  {c}: {n} unique", meta_csv[c].value_counts(dropna=False).to_dict() if n <= 12 else "")
t_ = next((r["CTD"].get("txt") for r in REC.values() if "CTD" in r and r["CTD"].get("txt")), None)
if t_: print("Transcript line-1 structure (masked):", re.sub(r"[A-Za-z]", "a", re.sub(r"\d", "9", open(dl(t_), encoding="utf-8", errors="ignore").readline()))[:120])

## 7. Dataset Column Mapping
The participant-ID, diagnosis and (optional) split columns of `meta-info.csv` are auto-detected (by name, then by content). Audio = `<folder>__CTD.wav`, transcript = `<folder>__CTD.txt`, task = CTD from the file name. Review the printout; override with `COLUMN_OVERRIDES={"id":..., "label":..., "split":...}` if wrong. Stops if the ID or diagnosis column cannot be found.

In [ ]:
import matplotlib.pyplot as plt
LAB = {0: "HC", 1: "MCI", 2: "Dementia"}; L2I = {v: k for k, v in LAB.items()}
def map_label(v):
    if v in LABEL_OVERRIDES: return LABEL_OVERRIDES[v]
    s = str(v).lower().strip()
    if "mci" in s or "mild" in s: return "MCI"
    if "dement" in s or "alzheimer" in s: return "Dementia"
    if s in ("hc", "control", "controls") or "healthy" in s or "control" in s: return "HC"
def pid_of(v):
    s = str(v).strip(); m_ = re.search(r"rec__?(\d+)", s) or re.fullmatch(r"(\d+)(?:\.0)?", s)
    return f"PROCESS-2_rec__{int(m_.group(1)):03d}" if m_ else None
cols = list(meta_csv.columns); ov = COLUMN_OVERRIDES
idc = ov.get("id") or next((c for c in cols if meta_csv[c].astype(str).str.contains("rec__").mean() > 0.9), None) \
      or next((c for c in cols if c.lower() in ["participant_id", "participant", "subject_id", "subject", "recording_id", "rec_id", "pid", "id", "name", "file", "recording"]
               and meta_csv[c].map(pid_of).notna().mean() > 0.9), None)
LABC = ["diagnosis", "dx", "label", "group", "class", "category", "cohort", "status", "condition", "diagnostic"]
labc = ov.get("label") or next((c for c in cols if c.lower() in LABC), None) \
       or next((c for c in cols if c != idc and meta_csv[c].nunique() <= 8 and len({map_label(v) for v in meta_csv[c].dropna().unique()} - {None}) >= 2), None)
spc = ov.get("split") or PARTITION_COLUMN or next((c for c in cols if c.lower() in ["split", "partition", "subset", "set", "fold", "train_test"]), None)
if not idc or not labc:
    raise RuntimeError(f"Could not identify id={idc}, label={labc} in meta-info.csv columns {cols}. Set COLUMN_OVERRIDES={{'id':..,'label':..}}.")
print("Mapping -> participant ID:", idc, "| diagnosis:", labc, "| split column:", spc or "none found")
meta_csv["participant"] = meta_csv[idc].map(pid_of)
if meta_csv.participant.isna().any(): raise RuntimeError(f"Column {idc!r} has values that cannot be matched to participant folders.")
assert (meta_csv.groupby("participant")[labc].nunique() <= 1).all(), "A participant has conflicting diagnoses in meta-info.csv"
if spc: assert (meta_csv.groupby("participant")[spc].nunique() <= 1).all(), "A participant has conflicting split values in meta-info.csv"
pmeta = meta_csv.drop_duplicates("participant").copy()
print("CSV rows:", len(meta_csv), "| unique participants:", len(pmeta), "| in CSV but no folder:", len(set(pmeta.participant) - set(REC)), "| folder but not in CSV:", len(set(REC) - set(pmeta.participant)))
# ---- counters for the verification report (section 11c) - computed from what is actually in the repository, never assumed ----
N_META = len(pmeta)
N_META_WAV = sum("wav" in REC.get(p, {}).get("CTD", {}) for p in pmeta.participant)
N_META_TXT = sum("txt" in REC.get(p, {}).get("CTD", {}) for p in pmeta.participant)
EXCL = {}        # exclusion reason -> number of participants (counts only, never IDs)

## 8. Label Mapping and Class Distribution
Raw diagnosis values are mapped to HC / MCI / Dementia. Any other value stops the run unless you add it to `LABEL_OVERRIDES` (mapping) or `DROP_LABELS` (exclude).

In [ ]:
pmeta["label_raw"] = pmeta[labc].astype(str)
RAW_MAP = {v: map_label(v) for v in sorted(pmeta.label_raw.unique())}; print("Raw label -> class:", RAW_MAP)
unm = [k for k, v in RAW_MAP.items() if v is None and k not in DROP_LABELS]
if unm: raise RuntimeError(f"Unmapped diagnosis values {unm}. Add to LABEL_OVERRIDES (-> HC/MCI/Dementia) or DROP_LABELS.")
n_drop = int(pmeta.label_raw.isin(DROP_LABELS).sum()); print("Excluded participants (DROP_LABELS):", n_drop)
if n_drop: EXCL["diagnosis value listed in DROP_LABELS"] = n_drop
pmeta = pmeta[~pmeta.label_raw.isin(DROP_LABELS)].copy(); pmeta["label"] = pmeta.label_raw.map(RAW_MAP); pmeta["label_id"] = pmeta.label.map(L2I)
print("Participants per class:", pmeta.label.value_counts().reindex(["HC", "MCI", "Dementia"]).to_dict())
pmeta.label.value_counts().reindex(["HC", "MCI", "Dementia"]).plot.bar(title="Participants per class"); plt.show()

## 9. CTD Task Filtering
CTD is chosen by file name (`…__CTD.wav` / `.txt`). SFT/PFT files are never touched. Transcripts are read now (small, with retries; a persistent download failure **stops** the run instead of silently becoming an "empty transcript"); audio is downloaded lazily during extraction. With `REQUIRE_COMPLETE_MODALITIES=True`, participants without a CTD recording or without a usable (non-empty) CTD transcript are excluded and counted – both inputs are required by the exported model, and an entirely missing modality is never imputed.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
rows = [dict(participant=p, wav=REC[p]["CTD"]["wav"], txt=REC[p]["CTD"].get("txt")) for p in pmeta.participant if "wav" in REC.get(p, {}).get("CTD", {})]
if not rows: raise RuntimeError("No CTD wav files found for any participant in meta-info.csv.")
EXCL["no CTD recording (wav) in the repository"] = len(pmeta) - len(rows)
ctd = pmeta.merge(pd.DataFrame(rows), on="participant").reset_index(drop=True); ctd["_row"] = np.arange(len(ctd))
def rd(f): return open(dl(f), encoding="utf-8", errors="ignore").read() if f else ""      # download errors RAISE (see dl); only a genuinely absent file gives ""
print("Reading CTD transcripts (small text files, parallel download)...")
with ThreadPoolExecutor(8) as ex: ctd["transcript"] = list(tqdm(ex.map(rd, ctd.txt), total=len(ctd), desc="transcripts"))
ctd["text_clean"] = ctd.transcript.map(pm.clean_text)
no_txt = ctd.txt.isna(); empty_txt = (~no_txt) & (ctd.text_clean.str.len() == 0)
EXCL["no CTD transcript file"] = int(no_txt.sum()); EXCL["empty CTD transcript after cleaning"] = int(empty_txt.sum())
print("CTD recordings:", len(ctd), "| participants lacking CTD wav:", len(pmeta) - len(ctd), "| missing transcript file:", int(no_txt.sum()), "| empty transcript:", int(empty_txt.sum()))
if REQUIRE_COMPLETE_MODALITIES:
    ctd = ctd[~(no_txt | empty_txt)].reset_index(drop=True)
else:
    print("WARNING: REQUIRE_COMPLETE_MODALITIES=False - participants with a missing transcript stay in and are median-imputed (not recommended for the final run).")
    EXCL["no CTD transcript file"] = EXCL["empty CTD transcript after cleaning"] = 0
print("Eligible participants with CTD audio + usable transcript:", ctd.participant.nunique())
print(ctd.label.value_counts().reindex(["HC", "MCI", "Dementia"]).to_dict())

## 10. Participant-Level Data Validation

In [ ]:
print("CTD recordings per participant:", ctd.groupby("participant").size().value_counts().to_dict(), "(one CTD wav per participant folder by file naming)")
print("Missing transcript file:", int(ctd.txt.isna().sum()), "| empty transcripts:", int((ctd.transcript.map(pm.clean_text).str.len() == 0).sum()))
assert ctd.participant.is_unique, "more than one CTD recording per participant - participant-level aggregation would be needed before splitting"
assert ctd.label.notna().all() and ctd.label_id.isin([0, 1, 2]).all(), "unlabelled participant in the cohort"

## 11. Train/Test Split Validation
If `meta-info.csv` has a split column (auto-detected or `PARTITION_COLUMN`; test values in `TEST_VALUES`) it is used as the **official** partition. Otherwise: participant-level stratified grouped 80/20 hold-out, clearly marked **GENERATED (not official)** – check the dataset card for an official split. The split is made at participant level, so all recordings of a participant stay in one partition. In `MODE="full"` no participant is dropped here: the debug restriction exists only in `MODE="debug"` and the code asserts this.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
ELIGIBLE_N = int(ctd.participant.nunique())                        # ALL eligible CTD participants, before any partitioning / debug restriction
ELIGIBLE_CLASS = ctd.groupby("label").participant.nunique().reindex(["HC", "MCI", "Dementia"]).fillna(0).astype(int).to_dict()
if spc:
    print("Split column values:", ctd[spc].astype(str).value_counts().to_dict())
    is_test = ctd[spc].astype(str).str.lower().isin([t.lower() for t in TEST_VALUES])
else: is_test = pd.Series(False, index=ctd.index)
if is_test.any() and (~is_test).any():
    ctd["partition"] = np.where(is_test, "test", "train"); SPLIT_IS_OFFICIAL = True
    SPLIT_STRATEGY = f"OFFICIAL partition from meta-info.csv column {spc!r}"
else:
    _, te_i = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(ctd, ctd.label_id, groups=ctd.participant))
    ctd["partition"] = "train"; ctd.loc[ctd.index[te_i], "partition"] = "test"; SPLIT_IS_OFFICIAL = False
    SPLIT_STRATEGY = "GENERATED (NOT official): participant-level stratified grouped 80/20 hold-out (no official split found in meta-info.csv)"
print("Split strategy:", SPLIT_STRATEGY)
if MODE == "debug":
    keep = ctd.groupby(["partition", "label"], group_keys=False).head(DEBUG_PER_CLASS).participant
    ctd = ctd[ctd.participant.isin(keep)].reset_index(drop=True)
    EXCL["DEBUG_PER_CLASS subset (debug mode only)"] = ELIGIBLE_N - int(ctd.participant.nunique())
    print("DEBUG subset:", ctd.participant.nunique(), "participants")
else:
    assert int(ctd.participant.nunique()) == ELIGIBLE_N, "FULL mode must use ALL eligible participants - a restriction was applied"
    print(f"FULL mode: all {ELIGIBLE_N} eligible participants kept (no debug restriction applied).")
for p in ["train", "test"]:
    d = ctd[ctd.partition == p]; print(p, "| recordings", len(d), "| participants", d.participant.nunique(), d.groupby("label").participant.nunique().to_dict())
ctd.groupby(["partition", "label"]).participant.nunique().unstack().plot.bar(title="Participants per class/partition"); plt.show()

## 11b. Participant-Level Leakage Check (PASS / FAIL)
Re-run again on the final analysed cohort in §18. Any FAIL stops the notebook before feature extraction / training.

In [ ]:
def leakage_report(df, title):
    train_participant_ids = set(df.loc[df.partition == "train", "participant"])
    test_participant_ids = set(df.loc[df.partition == "test", "participant"])
    checks = {
        "train and test participant IDs are disjoint": train_participant_ids.isdisjoint(test_participant_ids),
        "every participant is in exactly ONE partition": bool((df.groupby("participant").partition.nunique() == 1).all()),
        "one row per participant (participant-level samples, no recording can straddle partitions)": bool(df.participant.is_unique),
        "training partition contains all 3 classes": df.loc[df.partition == "train", "label_id"].nunique() == 3,
        "test partition is not empty": len(test_participant_ids) > 0,
    }
    print(f"--- LEAKAGE CHECK: {title} ---")
    for k, v in checks.items(): print(f"  [{'PASS' if v else 'FAIL'}] {k}")
    print(f"  train participants: {len(train_participant_ids)} | test participants: {len(test_participant_ids)}")
    if not all(checks.values()): raise RuntimeError("LEAKAGE CHECK FAILED - stopping before extraction/training.")
    assert set(train_participant_ids).isdisjoint(set(test_participant_ids))
    return "PASS"
leakage_report(ctd, "after split")

## 11c. Dataset Verification Report
Every number below is computed from the repository and `meta-info.csv` at run time (nothing is assumed). Counts only – no participant IDs. The report is printed again for the final analysed cohort in §18.

In [ ]:
def verification_report(df, title):
    tr_, te_ = df[df.partition == "train"], df[df.partition == "test"]
    cls = ["HC", "MCI", "Dementia"]; cc = lambda d: d.groupby("label").participant.nunique().reindex(cls).fillna(0).astype(int).to_dict()
    print("=" * 72); print(f"{title}   [MODE = {MODE.upper()}]"); print("=" * 72)
    print(f" 1. Participants in metadata (meta-info.csv) ........ {N_META}")
    print(f" 2. ... with a CTD recording (wav) ................... {N_META_WAV}")
    print(f" 3. ... with a CTD transcript file ................... {N_META_TXT}")
    print(f" 4. Eligible after filtering (CTD audio + transcript) {ELIGIBLE_N}")
    print(f"    Participants in THIS cohort ...................... {df.participant.nunique()}")
    c_ = cc(df); print(f" 5. HC ............. {c_['HC']}\n 6. MCI ............ {c_['MCI']}\n 7. Dementia ....... {c_['Dementia']}")
    print(f" 8. Excluded participants: {sum(EXCL.values())}")
    for k, v in EXCL.items(): print(f"      - {k}: {v}")
    print(f" 9. Training participants ... {tr_.participant.nunique()}\n10. Testing participants .... {te_.participant.nunique()}")
    print(f"11. Class distribution  train: {cc(tr_)}  |  test: {cc(te_)}")
    print(f"12. Split strategy: {SPLIT_STRATEGY}")
    if MODE == "debug": print("*** DEBUG SUBSET - not a full-dataset result ***")
    for c in cls:
        if cc(te_)[c] < MIN_TEST_PER_CLASS_WARN: print(f"WARNING: only {cc(te_)[c]} test participant(s) in class {c} (< {MIN_TEST_PER_CLASS_WARN}); test metrics for this class will be very unstable.")
    print("=" * 72)
verification_report(ctd, "VERIFICATION REPORT (after eligibility filtering and split)")

## 12. Audio Preprocessing
Loading (AudioDecoder / dict / bytes / path), mono conversion, resample to 16 kHz, peak normalisation, duration checks and 30 s chunking live in `predict_model3.py` (`load_audio`, `prep_audio`, `Encoders.audio_embedding`). Quick check on 3 recordings:

In [ ]:
WAV = dict(zip(ctd.participant, ctd.wav))
def get_audio(p): return dl(WAV[p])      # lazy download (with retries) -> local path
for p in ctd.participant.values[:3]:
    try:
        y0, sr0 = pm.load_audio(get_audio(p)); y1 = pm.prep_audio(y0, sr0)
        print(f"orig sr={sr0} dur={len(y0)/sr0:.1f}s -> {pm.SR} Hz dur={len(y1)/pm.SR:.1f}s, chunks={int(np.ceil(len(y1)/(pm.CHUNK_S*pm.SR)))}")
    except Exception as e: print("check failed on a recording:", type(e).__name__, "(will be handled/recorded during extraction)")

## 13. Wav2Vec2 Embedding Extraction
Each recording is decoded **once**; the same pass also computes the acoustic features of §14. Frozen encoder, 30 s chunks batched, length-weighted mean pooling. **Slowest cell – expected cost:** one download + one Wav2Vec2 forward pass + one librosa pYIN run per participant (pYIN is CPU-bound and is usually the bottleneck, even on GPU); the tqdm bar shows a live ETA.
**Safety of the cache:** results are cached per participant under `work/cache/<MODE>/<fingerprint>/`; the fingerprint hashes the mode, dataset commit, both encoder revisions and the feature-extraction code, so a cache can never be reused across incompatible configurations. The cache is **resumable** (atomic saves; re-run the cell after an interruption). If memory is short the code **halves the batch size and retries** – it never shrinks the dataset. Transient errors (download, out-of-memory) are never cached as failures; only genuinely unreadable/too-short/silent recordings are recorded as failed (and later excluded and reported, never filled with fake values).

In [ ]:
from tqdm.auto import tqdm
# ---------- cache fingerprint: caches are only reusable under an identical configuration ----------
FEATURE_CODE_SHA = hashlib.sha256(Path("predict_model3.py").read_bytes()).hexdigest()
FP = dict(mode=MODE, dataset=REPO, dataset_revision=DATASET_REV, task="CTD", wav2vec2=[pm.W2V_ID, REV[pm.W2V_ID]], distilbert=[pm.BERT_ID, REV[pm.BERT_ID]],
          feature_code_sha256=FEATURE_CODE_SHA, sampling_rate=pm.SR, chunk_s=pm.CHUNK_S, key="participant")
FP_ID = hashlib.sha256(json.dumps(FP, sort_keys=True).encode()).hexdigest()[:12]
CACHE_DIR = CACHE_ROOT / FP_ID
if RESET_CACHE and CACHE_DIR.exists(): shutil.rmtree(CACHE_DIR); print("cache reset")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
if (CACHE_DIR / "manifest.json").exists(): assert json.load(open(CACHE_DIR / "manifest.json")) == FP, "cache manifest does not match the current configuration"
else: json.dump(FP, open(CACHE_DIR / "manifest.json", "w"), indent=2)
print(f"cache: {MODE}/{FP_ID}")
print(f"""EXPECTED COST - audio pass: {len(ctd)} participants (minus cached). Per participant: download wav -> Wav2Vec2 ({DEVICE}, batch {BATCH_SIZE}) + librosa pYIN/MFCC (CPU).
  Free disk: {shutil.disk_usage(OUTPUT_DIR).free / 1e9:.1f} GB (downloaded WAVs live in the Hugging Face cache). Resumable: if interrupted, just re-run this cell.""")
enc = pm.Encoders(device=DEVICE, revisions=REV, batch_size=BATCH_SIZE)
def run_cached(name, fn, keys):
    path = CACHE_DIR / f"{name}.pkl"; cache = {}
    if path.exists():
        try:
            with open(path, "rb") as fh: cache = pickle.load(fh)
        except Exception as e:
            print(f"WARNING: cache {name} unreadable ({type(e).__name__}); moved aside and recomputing"); path.rename(path.with_suffix(".corrupt")); cache = {}
    todo = [k for k in keys if k not in cache]; print(f"{name}: {len([k for k in keys if k in cache])} cached, {len(todo)} to compute")
    def save():
        tmp = path.with_suffix(".tmp")
        with open(tmp, "wb") as fh: pickle.dump(cache, fh)
        os.replace(tmp, path)                      # atomic: an interrupted save can never corrupt the cache
    try:
        for n, k in enumerate(tqdm(todo, desc=name)):
            cache[k] = fn(k)
            if (n + 1) % 25 == 0: save()
    finally: save()
    return cache
def oom_safe(fn):
    """Run fn(); on out-of-memory halve the encoder batch size and retry (never drops data)."""
    while True:
        try: return fn()
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and enc.bs > 1:
                enc.bs = max(1, enc.bs // 2); print(f"out of memory -> encoder batch size reduced to {enc.bs}")
                if DEVICE == "cuda": torch.cuda.empty_cache()
                continue
            raise
TX = dict(zip(ctd.participant, ctd.transcript))
def audio_pass(p):
    path = get_audio(p)                            # download errors RAISE (not cached as a failure)
    try: y = pm.prep_audio(*pm.load_audio(path))
    except Exception as e: return {"err": re.sub(r"PROCESS-2_rec__\d+", "<id>", repr(e))[:150]}     # permanent data problem (unreadable / <0.5 s / silent)
    return {"emb": oom_safe(lambda: enc.audio_embedding(y)), "ac": pm.acoustic_features(y, TX[p])}
AP = run_cached("audio_pass", audio_pass, list(ctd.participant))
fails = [k for k in ctd.participant if "err" in AP[k]]; print("Failed recordings:", len(fails), "| distinct errors:", {AP[k]["err"] for k in fails})
X_AUD_REC = np.array([AP[k]["emb"] if "emb" in AP[k] else np.full(768, np.nan, np.float32) for k in ctd.participant])
print("Wav2Vec2 embedding matrix:", X_AUD_REC.shape)

## 14. Acoustic Feature Extraction
Features (computed in §13): duration, speaking ratio, silence, pause count/mean/rate (gaps ≥ 0.25 s between non-silent intervals, top_db=30), RMS stats, pYIN F0 stats, voiced fraction, 13 MFCC mean/std, words/min from transcript length. Voice-quality features (jitter/shimmer/HNR) are **not** included because they are not reliably computable with these tools. Unavailable values stay NaN.

In [ ]:
X_ACO_REC = pd.DataFrame([AP[k]["ac"] if "ac" in AP[k] else {} for k in ctd.participant]).reindex(columns=pm.ACOUSTIC_NAMES).values.astype(float)
miss = pd.Series(np.isnan(X_ACO_REC).mean(0), index=pm.ACOUSTIC_NAMES); print("Features with missing values (fraction):"); print(miss[miss > 0].round(3).to_dict())

## 15. Transcript Preprocessing
Minimal cleaning only (`clean_text`): removes bracketed annotations/tags and collapses whitespace; casing handled by the uncased tokenizer; filler words kept. Transcripts are never printed.

In [ ]:
ctd["text_clean"] = ctd.transcript.map(pm.clean_text)
wc = ctd.text_clean.str.split().str.len(); print("Empty after cleaning:", int((wc == 0).sum()), "| words/recording:", wc.describe()[["min", "50%", "max"]].to_dict())

## 16. DistilBERT Embedding Extraction
Frozen encoder; 512-token overlapping windows for long transcripts; attention-masked mean pooling. **Expected cost:** one forward pass per transcript (much cheaper than §13); cached per participant under the same fingerprinted cache folder and resumable; out-of-memory → batch size is halved, never the dataset.

In [ ]:
print(f"EXPECTED COST - DistilBERT pass: {len(ctd)} transcripts on {DEVICE} (cached + resumable; far cheaper than the audio pass).")
TXC = dict(zip(ctd.participant, ctd.text_clean))
TE = run_cached("text_emb", lambda p: oom_safe(lambda: enc.text_embedding(TXC[p])), list(ctd.participant))
X_TXT_REC = np.array([TE[k] if TE[k] is not None else np.full(768, np.nan, np.float32) for k in ctd.participant]); print(X_TXT_REC.shape)

## 17. Linguistic Feature Extraction
Word/sentence counts, TTR, MATTR-50, repeated words, fillers, pronoun ratio, mean word length, subordination ratio (a crude syntactic-complexity proxy; no parser).

In [ ]:
X_LNG_REC = pd.DataFrame([pm.linguistic_features(t) for t in ctd.text_clean]).reindex(columns=pm.LING_NAMES).values.astype(float)
print(X_LNG_REC.shape, "| NaN rows (empty transcripts):", int(np.isnan(X_LNG_REC).all(1).sum()))

## 18. Participant-Level Feature Aggregation and Final Cohort
Recording features are averaged (NaN-skipping) per participant, so one row = one participant and recordings can never straddle partitions. Participants whose CTD audio could not be used (unreadable, <0.5 s or silent) are **excluded and counted** (not imputed) when `REQUIRE_COMPLETE_MODALITIES=True`; the leakage check and verification report are then re-run on the final analysed cohort.

In [ ]:
OK = np.array(["emb" in AP[p] for p in ctd.participant]); n_bad = int((~OK).sum())
if REQUIRE_COMPLETE_MODALITIES and n_bad:
    EXCL["CTD audio unusable (unreadable / <0.5 s / silent), found during extraction"] = n_bad
    print(f"Excluding {n_bad} participant(s) without usable CTD audio (reported in the verification report).")
    X_AUD_REC, X_ACO_REC, X_TXT_REC, X_LNG_REC = (a[OK] for a in (X_AUD_REC, X_ACO_REC, X_TXT_REC, X_LNG_REC)); ctd = ctd[OK].reset_index(drop=True)
if MODE == "full" and REQUIRE_COMPLETE_MODALITIES: assert len(ctd) + n_bad == ELIGIBLE_N, "full-mode bookkeeping error: participants lost without being counted"
leakage_report(ctd, "final analysed cohort"); verification_report(ctd, "FINAL ANALYSED COHORT")
def agg(A):
    d = pd.DataFrame(A); d["p"] = ctd.participant.values; return d.groupby("p").mean()
P = ctd.groupby("participant").agg(label_id=("label_id", "first"), partition=("partition", "first"), n_rec=("_row", "size"))
assert P.index.is_unique and (P.n_rec == 1).all()
X_AUDIO = agg(X_AUD_REC).loc[P.index].values
X_ACOUSTIC = agg(X_ACO_REC).loc[P.index].values
X_TRANSCRIPT = agg(np.hstack([X_TXT_REC, X_LNG_REC])).loc[P.index].values
y = P.label_id.values.astype(int); groups = P.index.values; tr = (P.partition == "train").values; te = ~tr
ytr, gtr, yte = y[tr], groups[tr], y[te]
train_participant_ids, test_participant_ids = set(groups[tr]), set(groups[te])
assert set(train_participant_ids).isdisjoint(set(test_participant_ids)), "LEAKAGE: participant in both partitions"
TEXT_FEATURE_NAMES = [f"distilbert_{i}" for i in range(768)] + pm.LING_NAMES
print({k: v.shape for k, v in dict(audio=X_AUDIO, acoustic=X_ACOUSTIC, transcript=X_TRANSCRIPT).items()}, "| train/test participants:", tr.sum(), te.sum())
print("Participants with all-NaN audio / transcript embedding:", int(np.isnan(X_AUDIO).all(1).sum()), int(np.isnan(X_TRANSCRIPT[:, :768]).all(1).sum()))

## 19–22 setup: nested, fold-contained training utilities
Pipeline = median imputer → standard scaler → classifier, all fitted **inside** folds (so preprocessing never sees validation or test rows). Grid search (macro-F1) over LR / SVM / RandomForest with `StratifiedGroupKFold` on participant groups, **training partition only**. XGBoost is not used (small n; not justified).

**Validation design (nested / fold-contained).** The training partition is split into *outer* folds (identical for all four experiments). For every outer fold `k`, using **only the outer-training rows**: (i) each branch runs an *inner* grouped-CV grid search, is refit, and yields inner out-of-fold probabilities; (ii) the fusion step selects its variant (audio+transcript vs audio+acoustic+transcript) and `C` on those inner out-of-fold probabilities and is fitted on them; (iii) the outer-validation rows – never seen by any preprocessing, hyper-parameter search or fusion fit of that fold – are scored. Stacking the outer-validation predictions gives the **nested CV estimate** for all four experiments (this is what models are selected on). The **final** deployed models are produced by the same procedure on the whole training partition. The test partition is not touched until model selection is frozen (§23).
*Remaining limitation:* inside a fold, the inner out-of-fold probabilities that train the fusion classifier come from branch hyper-parameters chosen on those same rows (standard stacking practice; it does not leak the outer-validation rows).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_predict, ParameterGrid
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix,
                             roc_auc_score, classification_report, roc_curve, precision_recall_curve, auc)
min_cls = pd.Series(ytr).value_counts().min(); n_splits = int(min(N_SPLITS, min_cls))
assert len(set(ytr)) == 3 and n_splits >= 2, f"Training partition needs all 3 classes with >=2 participants each ({pd.Series(ytr).value_counts().to_dict()})"
CV = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)       # OUTER folds
OUTER = list(CV.split(np.zeros(len(ytr)), ytr, groups=gtr))                          # one fixed set of outer folds shared by ALL experiments
_cov = np.concatenate([b for _, b in OUTER]); assert len(_cov) == len(ytr) and len(set(_cov)) == len(ytr), "outer folds must partition the training rows"
for a, b in OUTER: assert set(gtr[a]).isdisjoint(gtr[b])
print("Outer CV folds:", n_splits)
def make_pipe(): return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("clf", LogisticRegression())])
GRID = [{"clf": [LogisticRegression(max_iter=3000, class_weight="balanced")], "clf__C": [1e-3, 1e-2, 1e-1, 1]},
        {"clf": [SVC(probability=True, class_weight="balanced", random_state=SEED)], "clf__C": [0.1, 1, 10], "clf__kernel": ["rbf", "linear"]},
        {"clf": [RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", min_samples_leaf=2, random_state=SEED, n_jobs=-1)], "clf__max_depth": [None, 5]}]
_ncfg = len(ParameterGrid(GRID)); _nin = int(min(INNER_SPLITS, np.bincount(ytr[OUTER[0][0]], minlength=3).min()))
print(f"EXPECTED COST - nested CV: 3 branches x ({n_splits} outer + 1 final) procedures x ({_ncfg} configs x {_nin} inner folds + {_nin + 1} extra fits) "
      f"~ {3 * (n_splits + 1) * (_ncfg * _nin + _nin + 1)} model fits (+ small fusion fits). Each fit takes seconds; progress bars below.")
def metrics(yt, proba):
    pred = proba.argmax(1); L = [0, 1, 2]
    p, r, f, _ = precision_recall_fscore_support(yt, pred, labels=L, zero_division=0)
    try: roc = float(roc_auc_score(yt, proba, multi_class="ovr", average="macro")) if len(set(yt)) == 3 else None
    except Exception: roc = None
    return dict(accuracy=accuracy_score(yt, pred), balanced_accuracy=balanced_accuracy_score(yt, pred), macro_precision=p.mean(), macro_recall=r.mean(),
                macro_f1=f.mean(), weighted_f1=f1_score(yt, pred, average="weighted", zero_division=0), roc_auc_ovr_macro=roc,
                per_class={LAB[i]: dict(precision=p[i], recall=r[i], f1=f[i]) for i in L}, confusion_matrix=confusion_matrix(yt, pred, labels=L).tolist())
def branch_procedure(X, y_, g):
    """Everything a branch is allowed to learn from (X, y_, g only): inner grouped-CV grid search -> refit -> inner out-of-fold probabilities."""
    n_in = int(min(INNER_SPLITS, np.bincount(y_, minlength=3).min()))
    if n_in < 2: raise RuntimeError(f"Too few participants of a class for inner CV (class counts {np.bincount(y_, minlength=3).tolist()})")
    cv_in = StratifiedGroupKFold(n_splits=n_in, shuffle=True, random_state=SEED)
    for a, b in cv_in.split(X, y_, groups=g): assert set(g[a]).isdisjoint(g[b])
    s = GridSearchCV(make_pipe(), GRID, scoring="f1_macro", cv=cv_in, refit=True, n_jobs=1, error_score="raise").fit(X, y_, groups=g)
    est = s.best_estimator_; assert list(est.classes_) == [0, 1, 2]
    oof = cross_val_predict(est, X, y_, cv=cv_in, groups=g, method="predict_proba")
    assert np.isfinite(oof).all() and np.allclose(oof.sum(1), 1.0), "invalid out-of-fold probabilities"
    return est, oof, {k: str(v) for k, v in s.best_params_.items()}, float(s.best_score_)
NEST, OOF, R, METR = {}, {}, {}, {}
def run_experiment(name, X):
    Xt = X[tr]; assert len(Xt) == len(ytr)                                  # ONLY training rows are ever used here
    oof_outer = np.full((len(ytr), 3), np.nan); NEST[name] = []
    for k, (a, b) in enumerate(tqdm(OUTER, desc=f"{name}: outer folds")):
        est, in_oof, prm, sc = branch_procedure(Xt[a], ytr[a], gtr[a])      # fold-contained selection
        val = est.predict_proba(Xt[b]); oof_outer[b] = val
        NEST[name].append(dict(a=a, b=b, in_oof=in_oof, val=val, params=prm, inner_f1=sc))
    assert np.isfinite(oof_outer).all()
    est, in_oof, prm, sc = branch_procedure(Xt, ytr, gtr)                    # FINAL model: same procedure on the whole training partition
    assert est.named_steps["sc"].n_samples_seen_ == len(ytr), "scaler must be fitted on training rows only"
    fold_f1 = [f1_score(ytr[b], oof_outer[b].argmax(1), average="macro") for _, b in OUTER]
    R[name] = dict(est=est, in_oof=in_oof, params=prm, inner_f1=sc); OOF[name] = oof_outer
    METR[name] = dict(train=metrics(ytr, est.predict_proba(Xt)), cv=metrics(ytr, oof_outer), cv_fold_macro_f1=fold_f1, inner_tuning_macro_f1=sc)
    print(f"{RESULT_TAG}[{name}] final params={R[name]['params']}\n   TRAIN (resubstitution) macro-F1={METR[name]['train']['macro_f1']:.3f} | "
          f"NESTED CV macro-F1={METR[name]['cv']['macro_f1']:.3f} (folds {np.mean(fold_f1):.3f} +/- {np.std(fold_f1):.3f}), acc={METR[name]['cv']['accuracy']:.3f}")

## 19. Experiment 1 – Audio Embedding Model (Wav2Vec2 → HC/MCI/Dementia)

In [ ]:
run_experiment('audio_embedding', X_AUDIO)

## 20. Experiment 2 – Acoustic Feature Model

In [ ]:
run_experiment('acoustic', X_ACOUSTIC)

## 21. Experiment 3 – Transcript Model (DistilBERT + linguistic features)

In [ ]:
run_experiment('transcript', X_TRANSCRIPT)

## 22. Experiment 4 – Late Fusion (proposed Model 3)
The fusion classifier is fitted only on **valid out-of-fold** branch probabilities (asserted finite, rows sum to 1, correct shape). Two variants (audio+transcript; audio+acoustic+transcript) and the fusion `C` are selected on those out-of-fold probabilities **inside every outer fold**; the outer-validation rows are only scored. The reported fusion CV estimate is therefore fully fold-contained. The final fusion model repeats the procedure on the whole training partition (branch models refit on the full training partition produce the test-time inputs). Fusion is **not** assumed to win – it competes with the three single-branch experiments in §23.

In [ ]:
VARIANTS = {"audio+transcript": ["audio_embedding", "transcript"], "audio+acoustic+transcript": ["audio_embedding", "acoustic", "transcript"]}
FUSION_GRID = {"clf__C": [0.01, 0.1, 1, 10]}
def fusion_procedure(Z, y_, g):
    """Z: branch -> inner out-of-fold probabilities for exactly the rows (y_, g). Selects variant + C by grouped CV, returns the fitted fusion pipeline."""
    n_in = int(min(INNER_SPLITS, np.bincount(y_, minlength=3).min())); cv_in = StratifiedGroupKFold(n_splits=n_in, shuffle=True, random_state=SEED)
    best, scores = None, {}
    for vn, ins in VARIANTS.items():
        Zv = np.hstack([Z[b] for b in ins]); assert Zv.shape == (len(y_), 3 * len(ins)) and np.isfinite(Zv).all(), "fusion inputs must be valid out-of-fold probabilities"
        s = GridSearchCV(Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))]), FUSION_GRID,
                         scoring="f1_macro", cv=cv_in, error_score="raise").fit(Zv, y_, groups=g)
        scores[vn] = float(s.best_score_)
        if best is None or s.best_score_ > best[1]: best = (vn, s.best_score_, s)      # ties -> earlier (simpler) variant
    return best[0], best[2].best_estimator_, {k: str(v) for k, v in best[2].best_params_.items()}, scores
BR = ["audio_embedding", "acoustic", "transcript"]; assert all(b in NEST for b in BR), "run Experiments 1-3 first"
fus_oof = np.full((len(ytr), 3), np.nan); fold_variants = []
for k, (a, b) in enumerate(tqdm(OUTER, desc="fusion: outer folds")):
    for bn in BR: assert np.array_equal(NEST[bn][k]["b"], b) and np.array_equal(NEST[bn][k]["a"], a)     # same outer folds for every branch
    vn, fest, _, _ = fusion_procedure({bn: NEST[bn][k]["in_oof"] for bn in BR}, ytr[a], gtr[a])           # fitted on outer-TRAIN inner-OOF only
    fus_oof[b] = fest.predict_proba(np.hstack([NEST[bn][k]["val"] for bn in VARIANTS[vn]])); fold_variants.append(vn)
assert np.isfinite(fus_oof).all()
vn, fest, prm, vscores = fusion_procedure({bn: R[bn]["in_oof"] for bn in BR}, ytr, gtr)                  # FINAL fusion model
assert fest.named_steps["sc"].n_samples_seen_ == len(ytr)
R["fusion"] = dict(est=fest, inputs=VARIANTS[vn], variant=vn, params=prm, variant_inner_cv_f1=vscores); OOF["fusion"] = fus_oof
ffold = [f1_score(ytr[b], fus_oof[b].argmax(1), average="macro") for _, b in OUTER]
METR["fusion"] = dict(train=metrics(ytr, fest.predict_proba(np.hstack([R[bn]["in_oof"] for bn in VARIANTS[vn]]))), cv=metrics(ytr, fus_oof), cv_fold_macro_f1=ffold, variants_chosen_per_outer_fold=fold_variants)
print(f"{RESULT_TAG}fusion: NESTED CV macro-F1={METR['fusion']['cv']['macro_f1']:.3f} (folds {np.mean(ffold):.3f} +/- {np.std(ffold):.3f}) | variant per outer fold: {dict(Counter(fold_variants))}")
print("Final fusion variant:", vn, "| inputs:", R["fusion"]["inputs"], "| params:", prm)
print("(train metric for fusion = on its own out-of-fold inputs; it is not a resubstitution score of the branch models)")

## 23. Cross-Validation Summary, Leakage Checks and Model Selection
Model selection is frozen here using **nested CV macro-F1 on the training partition only** – before the test partition is evaluated. Fusion is chosen only if it actually has the best CV score (ties go to the simpler single-branch model).

In [ ]:
for i, (a, b) in enumerate(OUTER):
    assert not set(gtr[a]) & set(gtr[b]); print(f"outer fold {i}: train classes {np.bincount(ytr[a], minlength=3)} | val classes {np.bincount(ytr[b], minlength=3)} | participant overlap 0")
assert set(groups[tr]).isdisjoint(set(groups[te])); print("Train/test participant overlap: 0")
banned = ("participant", "label", "file", "diagnos", "age", "gender", "mmse")
assert not [n for n in pm.ACOUSTIC_NAMES + TEXT_FEATURE_NAMES if any(w == n.lower() or n.lower().startswith(w + "_") for w in banned)]
for bn in BR: assert isinstance(R[bn]["est"], Pipeline) and R[bn]["est"].named_steps["sc"].n_samples_seen_ == int(tr.sum())
print("[PASS] no ID/label/file/clinical-score features | [PASS] imputer/scaler fitted on training rows only (inside every fold) | [PASS] test set untouched so far.")
order = ["audio_embedding", "acoustic", "transcript", "fusion"]
CVT = pd.DataFrame({n: {"cv_macro_f1": METR[n]["cv"]["macro_f1"], "cv_fold_std": float(np.std(METR[n]["cv_fold_macro_f1"])), "cv_accuracy": METR[n]["cv"]["accuracy"],
                        "cv_balanced_acc": METR[n]["cv"]["balanced_accuracy"], "train_macro_f1(resub)": METR[n]["train"]["macro_f1"]} for n in order}).T
print(RESULT_TAG + "NESTED CV (training partition) - all four experiments:"); print(CVT.round(3))
top = CVT.cv_macro_f1.max(); SELECTED = next(n for n in order if CVT.cv_macro_f1[n] >= top - 1e-12)
BEST_BRANCH = CVT.loc[BR].cv_macro_f1.idxmax(); runner = CVT.cv_macro_f1.drop(SELECTED).max()
print("\nSELECTED (by nested CV macro-F1, before any test evaluation):", SELECTED)
print("Best single branch (CV):", BEST_BRANCH, "| fusion beats best branch in CV:", bool(CVT.loc["fusion", "cv_macro_f1"] > CVT.loc[BEST_BRANCH, "cv_macro_f1"]))
if top - runner < CVT.loc[SELECTED, "cv_fold_std"]: print("NOTE: the selected model's CV margin over the runner-up is smaller than its fold-to-fold std - the top candidates are statistically hard to distinguish.")
SELECTION_FROZEN = True; SELECTION_RECORD = dict(selected=SELECTED, cv_macro_f1={n: float(CVT.cv_macro_f1[n]) for n in order})   # no re-selection after this point

## 24. Final Model Evaluation (test partition)
All four experiments are reported once on the untouched test partition (official split if one exists in `meta-info.csv`, otherwise the **generated** hold-out – the output says which). Test predictions are computed here for the first time. No retuning or re-selection happens after this point; the final model is the one frozen in §23. Warnings are printed when a test class has few participants; ROC-AUC is not reported if a class is absent.

In [ ]:
assert globals().get("SELECTION_FROZEN"), "model selection must be frozen on CV results before the test partition is touched"
TEST_NAME = "OFFICIAL TEST PARTITION" if SPLIT_IS_OFFICIAL else "GENERATED HOLD-OUT (not an official split)"
XS = dict(audio_embedding=X_AUDIO, acoustic=X_ACOUSTIC, transcript=X_TRANSCRIPT)
for bn in BR: R[bn]["test_proba"] = R[bn]["est"].predict_proba(XS[bn][te])
R["fusion"]["test_proba"] = R["fusion"]["est"].predict_proba(np.hstack([R[bn]["test_proba"] for bn in R["fusion"]["inputs"]]))
for n in R: METR[n]["test"] = metrics(yte, R[n]["test_proba"])
TEST_COUNTS = np.bincount(yte, minlength=3); print(RESULT_TAG + f"{TEST_NAME} class counts:", TEST_COUNTS, "(HC, MCI, Dementia)")
for c in range(3):
    if TEST_COUNTS[c] < MIN_TEST_PER_CLASS_WARN: print(f"WARNING: test class {LAB[c]} has only {TEST_COUNTS[c]} participant(s) - per-class metrics are very unstable" + (" and ROC-AUC is unavailable." if TEST_COUNTS[c] == 0 else "."))
REPORTS = {}
for n in order:
    pr = R[n]["test_proba"].argmax(1); REPORTS[n] = classification_report(yte, pr, labels=[0, 1, 2], target_names=list(LAB.values()), zero_division=0, output_dict=True)
    print(f"\n=== {RESULT_TAG}{n} ({TEST_NAME}){'   <-- SELECTED' if n == SELECTED else ''} ===\n" + classification_report(yte, pr, labels=[0, 1, 2], target_names=list(LAB.values()), zero_division=0))
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, n in zip(ax, order):
    cm = np.array(METR[n]["test"]["confusion_matrix"]); a.imshow(cm, cmap="Blues"); a.set_title(n); a.set_xticks(range(3), LAB.values()); a.set_yticks(range(3), LAB.values())
    for i in range(3):
        for j in range(3): a.text(j, i, cm[i, j], ha="center", va="center")
    a.set_xlabel("Predicted"); a.set_ylabel("True")
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(2, 4, figsize=(18, 7))
for k, n in enumerate(order):
    for c in range(3):
        if (yte == c).sum() == 0 or (yte != c).sum() == 0: continue
        fpr, tpr, _ = roc_curve(yte == c, R[n]["test_proba"][:, c]); ax[0, k].plot(fpr, tpr, label=f"{LAB[c]} AUC={auc(fpr, tpr):.2f}")
        pp, rr, _ = precision_recall_curve(yte == c, R[n]["test_proba"][:, c]); ax[1, k].plot(rr, pp, label=LAB[c])
    ax[0, k].plot([0, 1], [0, 1], "k--"); ax[0, k].set_title(f"ROC (OvR) {n}"); ax[1, k].set_title(f"PR {n}"); ax[0, k].legend(); ax[1, k].legend()
plt.tight_layout(); plt.show()
TRAINING_OK = True; EVAL_OK = True

## 25. Model Comparison and Selection
Cross-validation performance (nested, training partition) and test performance are kept in separate columns; only the CV column drove model selection.

In [ ]:
CMP = pd.DataFrame({n: {"cv_macro_f1": METR[n]["cv"]["macro_f1"], "cv_balanced_acc": METR[n]["cv"]["balanced_accuracy"], "test_accuracy": METR[n]["test"]["accuracy"], "test_balanced_acc": METR[n]["test"]["balanced_accuracy"],
                        "test_macro_precision": METR[n]["test"]["macro_precision"], "test_macro_recall": METR[n]["test"]["macro_recall"],
                        "test_macro_f1": METR[n]["test"]["macro_f1"], "test_weighted_f1": METR[n]["test"]["weighted_f1"], "test_roc_auc_ovr": METR[n]["test"]["roc_auc_ovr_macro"]} for n in order}).T
CMP.index.name = "experiment"; print(RESULT_TAG + f"cv_* = nested CV on training partition | test_* = {TEST_NAME}"); print(CMP.astype(float).round(3))
print("\nPer-class TEST (precision/recall/F1):")
print(pd.DataFrame({(n, c): METR[n]["test"]["per_class"][c] for n in order for c in LAB.values()}).T.round(3))
print(f"\nFinal model = {SELECTED} (chosen by nested CV before test evaluation).")
print("Fusion vs best branch on TEST macro-F1:", round(METR['fusion']['test']['macro_f1'], 3), "vs", round(METR[BEST_BRANCH]['test']['macro_f1'], 3), f"({BEST_BRANCH}); test-set differences are NOT used to re-select.")

## 26. Model Export
Exports fitted classifiers, preprocessing configs, metrics and the inference script into `neurodetect_model3_process2_package/` (a `MODE="debug"` run writes `..._DEBUG_package` instead and can never produce the final package). Encoders are **not** bundled: the inference script downloads the exact pinned Hugging Face revisions (so the package is **not** fully offline). Refuses to export if training or test evaluation did not complete. Counts only are exported – no participant IDs, recordings, transcripts, dataset files or tokens.

In [ ]:
assert globals().get("TRAINING_OK") and globals().get("EVAL_OK"), "Training/evaluation did not complete - export aborted (no package created)."
import joblib, importlib.metadata as im
PKG_NAME = "neurodetect_model3_process2_package" + ("" if MODE == "full" else "_DEBUG")
PKG = OUTPUT_DIR / PKG_NAME
if PKG.exists(): shutil.rmtree(PKG)
for d in ["models", "preprocessing", "evaluation", "inference"]: (PKG / d).mkdir(parents=True)
FILES_OUT = {"audio_embedding": "audio_embedding_classifier.joblib", "acoustic": "acoustic_classifier.joblib", "transcript": "transcript_classifier.joblib", "fusion": "fusion_classifier.joblib"}
for k, f in FILES_OUT.items(): joblib.dump(R[k]["est"], PKG / "models" / f)
def J(path, obj): json.dump(obj, open(path, "w"), indent=2, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
def _ver(p):
    try: return im.version(p)
    except Exception: return None
VERS = {p: _ver(p) for p in ["numpy", "scipy", "scikit-learn", "joblib", "librosa", "soundfile", "torch", "transformers", "huggingface_hub"] if _ver(p)}
MODEL_VERSION = f"model3-process2-ctd-{time.strftime('%Y%m%d')}" + ("-full" if MODE == "full" else "-DEBUG")
J(PKG / "preprocessing/audio_feature_config.json", dict(sampling_rate=pm.SR, chunk_seconds=pm.CHUNK_S, min_tail_seconds=pm.MIN_TAIL_S, wav2vec2_id=pm.W2V_ID, wav2vec2_revision=REV[pm.W2V_ID],
  pooling="mean over time, length-weighted over 30s chunks", normalisation="peak", acoustic_feature_names=pm.ACOUSTIC_NAMES, silence_top_db=pm.TOP_DB, pause_min_seconds=pm.PAUSE_MIN_S, f0="librosa.pyin 65-400 Hz", aggregation="participant mean (NaN-skipping)"))
J(PKG / "preprocessing/linguistic_feature_config.json", dict(distilbert_id=pm.BERT_ID, distilbert_revision=REV[pm.BERT_ID], max_length=512, stride=64, pooling="attention-masked mean",
  linguistic_feature_names=pm.LING_NAMES, fillers=sorted(pm.FILLERS), pronouns=sorted(pm.PRONOUNS), subordinators=sorted(pm.SUBORD), text_feature_order="768 distilbert dims then linguistic features"))
J(PKG / "preprocessing/label_mapping.json", dict(id_to_label=LAB, label_to_id=L2I, dataset_raw_label_map=RAW_MAP))
J(PKG / "preprocessing/model_config.json", dict(model_version=MODEL_VERSION, mode=MODE, final_model=SELECTED, fusion_inputs=R["fusion"]["inputs"], fusion_variant=R["fusion"]["variant"], model_files=FILES_OUT,
  encoder_revisions=REV, acoustic_feature_names=pm.ACOUSTIC_NAMES, linguistic_feature_names=pm.LING_NAMES, task="CTD", seed=SEED,
  best_params={k: R[k]["params"] for k in R}, outer_cv_folds=n_splits, inner_cv_folds=INNER_SPLITS,
  feature_order=dict(audio_embedding="768 Wav2Vec2 dims", acoustic="ACOUSTIC_NAMES order", transcript="768 DistilBERT dims then LING_NAMES order", fusion="concatenated branch probabilities in fusion_inputs order, classes [HC, MCI, Dementia]")))
J(PKG / "evaluation/metrics.json", dict(mode=MODE, test_partition=TEST_NAME, note="cv = nested CV on the training partition (used for selection); test = evaluated once after selection was frozen; train = resubstitution (optimistic)", metrics=METR))
J(PKG / "evaluation/classification_report.json", REPORTS)
J(PKG / "evaluation/confusion_matrix.json", {n: dict(test=METR[n]["test"]["confusion_matrix"], cv=METR[n]["cv"]["confusion_matrix"], labels=list(LAB.values())) for n in order})
CMP.to_csv(PKG / "evaluation/model_comparison.csv")
shutil.copy("predict_model3.py", PKG / "inference/predict_model3.py")
assert hashlib.sha256((PKG / "inference/predict_model3.py").read_bytes()).hexdigest() == FEATURE_CODE_SHA, "inference script differs from the code used for feature extraction"
open(PKG / "inference/requirements.txt", "w").write("\n".join(f"{p}=={v.split('+')[0]}" for p, v in VERS.items()) + "\n")
cls = ["HC", "MCI", "Dementia"]
J(PKG / "model_metadata.json", dict(model_version=MODEL_VERSION, mode=MODE, dataset="CognoSpeak/PROCESS-2 (CTD only; not redistributed)", dataset_revision=DATASET_REV,
  split_strategy=SPLIT_STRATEGY, split_is_official=bool(SPLIT_IS_OFFICIAL), validation_scheme="nested grouped CV: outer folds on the training partition; branch hyper-parameters, fusion variant and fusion C selected inside each outer fold; test partition evaluated once after selection was frozen",
  n_participants_in_metadata=N_META, total_eligible_participants=ELIGIBLE_N, n_analysed_participants=int(len(P)), exclusions_by_reason=EXCL,
  n_train_participants=int(tr.sum()), n_test_participants=int(te.sum()),
  train_class_distribution=dict(zip(cls, np.bincount(ytr, minlength=3).tolist())), test_class_distribution=dict(zip(cls, np.bincount(yte, minlength=3).tolist())),
  selected_model=SELECTED, fusion_variant=R["fusion"]["variant"], fusion_inputs=R["fusion"]["inputs"], selection_record=SELECTION_RECORD,
  encoder_revisions=REV, dependency_versions=VERS, python=platform.python_version(), device_used=DEVICE,
  evaluation_metrics=METR, selected_model_cv_metrics=METR[SELECTED]["cv"], selected_model_test_metrics=METR[SELECTED]["test"], disclaimer=pm.DISCLAIMER))
NOTE = "" if MODE == "full" else "**DEBUG-MODE package: trained on a tiny subset, do not use.**"
open(PKG / "README.md", "w").write(f"""# NeuroDetect Model 3 (PROCESS-2, CTD) - {MODEL_VERSION}
{pm.DISCLAIMER}

Research/education use only, subject to the PROCESS-2 Data Use Agreement. Contains no recordings, transcripts, participant IDs, dataset files or tokens
(SVM/RF files hold derived feature vectors only). {NOTE}

Training mode: **{MODE}** | participants analysed: {len(P)} (train {int(tr.sum())}, test {int(te.sum())}) | split: {SPLIT_STRATEGY}
Selected model: **{SELECTED}** (fusion inputs: {R['fusion']['inputs']}). Classes: HC, MCI, Dementia. Input: CTD audio + transcript.
Metrics in `evaluation/`: `cv` = nested CV on the training partition (used for selection), `test` = evaluated once after selection was frozen, `train` = resubstitution (optimistic).

Usage: `pip install -r inference/requirements.txt`, then
```python
import sys; sys.path.insert(0, "inference")
from predict_model3 import NeuroDetectModel3
m = NeuroDetectModel3("path/to/package"); print(m.predict(audio_path="sample.wav", transcript="..."))
```
NOT fully offline - required Hugging Face downloads at the pinned revisions in `preprocessing/model_config.json`:
`{pm.W2V_ID}` @ `{REV[pm.W2V_ID]}` and `{pm.BERT_ID}` @ `{REV[pm.BERT_ID]}` (pretrained weights are not bundled; check their model cards for licences).
""")
if MODE == "full":
    assert "debug" not in json.dumps(json.load(open(PKG / "model_metadata.json"))).lower() and "debug" not in open(PKG / "README.md").read().lower(), "full package must not carry DEBUG markers"
print("Package built at", PKG)

## 27. Inference Test and Preprocessing-Parity Check
1. **Smoke test** – a *synthetic tone + dummy sentence* (no PROCESS-2 data) only checks that the exported package runs end-to-end (the prediction is meaningless). Set `TEST_AUDIO` to your own non-sensitive WAV to test realistically.
2. **Parity check** – three *training-partition* participants are pushed through the exported inference code (raw audio + transcript → features → exported models). Branch/final probabilities must match the in-memory training pipeline, proving identical preprocessing and feature ordering. Test-partition participants are not used. Export is blocked if this fails.

In [ ]:
import soundfile as sf, tempfile
TEST_AUDIO, TEST_TEXT = None, "there is a boy on a stool and the sink is overflowing while the mother is washing dishes"
if TEST_AUDIO is None:
    t = np.linspace(0, 6, 6 * 16000, endpoint=False); TEST_AUDIO = str(Path(tempfile.mkdtemp()) / "synthetic.wav")
    sf.write(TEST_AUDIO, (0.3 * np.sin(2 * np.pi * 180 * t) * (1 + np.sin(2 * np.pi * 3 * t))).astype("float32"), 16000)
    print("Using SYNTHETIC audio (pipeline smoke test only).")
M3 = pm.NeuroDetectModel3(PKG, device=DEVICE, batch_size=BATCH_SIZE)
print("Required inputs:", M3.required_inputs())
print(json.dumps(M3.predict(audio_path=TEST_AUDIO, transcript=TEST_TEXT), indent=2))
print("Missing-input check:", M3.predict(audio_path=None, transcript=None)["errors"])
# ---- parity: exported inference path vs in-memory training pipeline (training participants only) ----
assert M3.cfg["acoustic_feature_names"] == pm.ACOUSTIC_NAMES and M3.cfg["linguistic_feature_names"] == pm.LING_NAMES, "feature order in the exported config differs from training"
XB = dict(audio_embedding=X_AUDIO, acoustic=X_ACOUSTIC, transcript=X_TRANSCRIPT); worst = 0.0
for i in np.where(tr)[0][:3]:
    p = P.index[i]; out = M3.predict(audio_path=get_audio(p), transcript=TX[p]); assert out["status"] == "ok", out
    exp = {b: R[b]["est"].predict_proba(XB[b][i:i + 1])[0] for b in M3._branches()}
    for b in M3._branches(): worst = max(worst, float(np.abs(np.array([out["branch_probabilities"][b][LAB[c]] for c in range(3)]) - exp[b]).max()))
    if SELECTED == "fusion":
        ef = R["fusion"]["est"].predict_proba(np.hstack([exp[b] for b in R["fusion"]["inputs"]])[None])[0]
        worst = max(worst, float(np.abs(np.array([out["probabilities"][LAB[c]] for c in range(3)]) - ef).max()))
print(f"Parity check on 3 training participants: max |probability difference| = {worst:.2e}")
assert worst < 5e-3, "PARITY FAILED: exported inference path does not reproduce the training pipeline - export blocked"
PARITY_OK = True; print("[PASS] inference preprocessing and feature ordering match training")

## 28. Verify and Download Model Package
Verifies the package (all expected files, metadata says `full`, no DEBUG markers, no raw audio/transcripts/IDs/tokens), prints size and listing, then triggers the Colab download of `neurodetect_model3_process2_package.zip`. Nothing is built or downloaded if training, evaluation or the parity check failed.

In [ ]:
assert globals().get("TRAINING_OK") and globals().get("EVAL_OK") and globals().get("PARITY_OK") and PKG.exists(), "Export did not complete / checks failed - no ZIP."
ZIP = OUTPUT_DIR / (PKG.name + ".zip")
if ZIP.exists(): ZIP.unlink()
shutil.make_archive(str(ZIP.with_suffix("")), "zip", root_dir=OUTPUT_DIR, base_dir=PKG.name)
EXPECT = ["models/audio_embedding_classifier.joblib", "models/acoustic_classifier.joblib", "models/transcript_classifier.joblib", "models/fusion_classifier.joblib",
          "preprocessing/audio_feature_config.json", "preprocessing/linguistic_feature_config.json", "preprocessing/label_mapping.json", "preprocessing/model_config.json",
          "evaluation/metrics.json", "evaluation/classification_report.json", "evaluation/confusion_matrix.json", "evaluation/model_comparison.csv",
          "inference/predict_model3.py", "inference/requirements.txt", "README.md", "model_metadata.json"]
zf = zipfile.ZipFile(ZIP); names = [n for n in zf.namelist() if not n.endswith("/")]
missing = [f for f in EXPECT if f"{PKG.name}/{f}" not in names]; assert not missing, f"ZIP is missing {missing}"
meta = json.loads(zf.read(f"{PKG.name}/model_metadata.json"))
if MODE == "full":
    assert ZIP.name == "neurodetect_model3_process2_package.zip" and meta["mode"] == "full" and "debug" not in meta["model_version"].lower(), "not a full-mode package"
    assert "debug" not in json.dumps(meta).lower(), "DEBUG marker found in metadata"
    assert meta["n_train_participants"] + meta["n_test_participants"] == meta["n_analysed_participants"] and meta["total_eligible_participants"] >= meta["n_analysed_participants"]
BAD_EXT = (".wav", ".flac", ".mp3", ".m4a", ".ogg", ".parquet", ".pkl", ".npy", ".npz", ".pt", ".bin", ".safetensors", ".h5")
assert not [n for n in names if n.lower().endswith(BAD_EXT)], "Raw data / weights found in package!"
assert not [n for n in names if n.lower().endswith((".txt", ".csv")) and not n.endswith(("inference/requirements.txt", "evaluation/model_comparison.csv"))], "Unexpected transcript/data file in package!"
for n in names:                                    # content scan: no participant IDs, no Hugging Face tokens in ANY file
    b = zf.read(n); assert not re.search(rb"PROCESS-2_rec__\d+", b), f"participant ID found in {n}"; assert not re.search(rb"hf_[A-Za-z0-9]{20,}", b), f"token-like string found in {n}"
print(f"ZIP: {ZIP}  ({ZIP.stat().st_size / 1e6:.2f} MB) | mode={meta['mode']} | model={meta['model_version']} | selected={meta['selected_model']}")
for i in zf.infolist():
    if not i.filename.endswith("/"): print(f"  {i.file_size / 1e3:10.1f} kB  {i.filename}")
if MODE != "full": print("\nDEBUG package verified but NOT offered for download (it is not the final package). Set MODE='full' for the final run.")
elif IN_COLAB:
    from google.colab import files; files.download(str(ZIP))
else: print("\nNot running in Colab - copy the ZIP from:", ZIP)

## 29. Final Research Summary and Limitations
Results are only those printed above from your own run; published PROCESS-2 benchmarks are **not** ours. Cross-validation numbers (nested, training partition) and test numbers are reported separately; only the former drove model selection. Check the printed split strategy: if it says *GENERATED*, the test partition is a generated hold-out, not an official PROCESS-2 split.

**Limitations:** small participant counts (especially Dementia) → wide uncertainty and unstable per-class estimates (see test-class warnings); class imbalance; CTD-only, dataset-specific recording conditions and prompts; accent/language variation; unknown generalisation to external populations and domain shift (e.g. app microphones, ASR vs verified transcripts); nested CV removes the optimism from tuning on the validation fold, but inside each fold the fusion classifier is trained on out-of-fold probabilities of branch models whose hyper-parameters were chosen on the same rows (standard stacking; no outer-validation leakage), and with few participants even nested estimates carry large variance; single grouped CV partitioning (no repeated CV); no subgroup (age/gender) conclusions are possible at this sample size; participants without usable audio/transcript are excluded (counts reported) rather than imputed; SFT/PFT are future extensions. Per the DUA, delete the local Hugging Face cache (`~/.cache/huggingface`) and the `work/` folder when finished.
This is research screening, not clinical diagnosis. **NeuroDetect Model 3 is an experimental speech-based research screening model. Its predictions are not a medical diagnosis and must not be used as a substitute for professional clinical assessment.**